# A score of 0.6 that means 0.6

MichAl Academy, lesson 2.8.

Run each cell with **Shift+Enter**.

Lesson 2.7 left you with a threshold and no way to choose it. This notebook
answers two questions that turn out to be one question.

First: when a model says 0.6, do 60% of those cases turn out positive? For one
of the models here the answer is no, and it is the model with the best ROC-AUC
of the lot.

Second: given that a missed detection costs more than a false alarm, where does
the threshold belong? There is a one-line formula for it, and the formula is
only trustworthy if the answer to the first question is yes.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score, accuracy_score

SEED = 0
digits = load_digits()
X = digits.data
y = (digits.target == 8).astype(int)

# Every score in this notebook is out of fold: each image is scored by a model
# that did not train on it. Lesson 2.3's rule, and it matters more here than
# anywhere, because a model scoring its own training data looks perfectly
# calibrated and is telling you nothing.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"{len(y)} images, {y.sum()} of them 8s, base rate {y.mean():.4f}")


## 1. Reading a reliability table

Take every image the model scored between 0.5 and 0.7, and ask what fraction of
them really were 8s. If the score means what it says, that fraction is about
0.6. Do it for six bands and you have a reliability table.

The bands are fixed width rather than equal population. On a problem where only
9.7% of cases are positive, equal-population bands put four of the six below
0.07 and hide the only end anyone thresholds on.


In [ ]:
EDGES = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0001]


def reliability(y_true, p, label):
    print(f"\n{label}")
    print(f"  Brier {brier_score_loss(y_true, p):.4f}"
          f"   ROC-AUC {roc_auc_score(y_true, p):.4f}"
          f"   accuracy at 0.5 {accuracy_score(y_true, p >= 0.5):.4f}")
    print(f"  {'band':>12} {'n':>6} {'says':>7} {'really':>8} {'gap':>7}")
    for lo, hi in zip(EDGES[:-1], EDGES[1:]):
        m = (p >= lo) & (p < hi)
        if not m.any():
            print(f"  {f'{lo:.1f}-{hi:.1f}':>12} {'empty':>6}")
            continue
        says, really = p[m].mean(), y_true[m].mean()
        print(f"  {f'{lo:.1f}-{min(hi,1.0):.1f}':>12} {int(m.sum()):>6}"
              f" {says:>7.3f} {really:>8.3f} {really - says:>+7.3f}")


def out_of_fold(estimator):
    return cross_val_predict(estimator, X, y, cv=cv, method="predict_proba")[:, 1]


forest = RandomForestClassifier(random_state=SEED, n_jobs=1)
p_forest = out_of_fold(forest)
reliability(y, p_forest, "random forest, straight out of the box")


Read the `gap` column from the bottom up.

The forest said **0.380** for 53 images and **69.8%** of them were 8s. It said
**0.602** for 47 images and **every single one** was an 8. It said 0.796 for 63
images, all 8s. It said 0.933 for 7 images, all 8s.

Those gaps are not noise. If the truth behind that 0.5-0.7 band really were
0.602, then all 47 of 47 coming back as 8s has probability 0.602 to the power
47, which is about four in a hundred billion.

So what is the forest's score actually measuring? The next cell answers it
rather than asserting it.


In [ ]:
# A random forest's default min_samples_leaf is 1, so its leaves are pure.
# Check what that means for what a single tree can say, and what the ensemble
# number therefore is.
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          random_state=SEED, stratify=y)
one_forest = RandomForestClassifier(random_state=SEED, n_jobs=1).fit(X_tr, y_tr)

per_tree = np.stack([t.predict_proba(X_te)[:, 1] for t in one_forest.estimators_])
print(f"values a single tree ever outputs: {np.unique(per_tree)}")

vote_share = per_tree.mean(axis=0)
ensemble = one_forest.predict_proba(X_te)[:, 1]
print(f"max difference between the forest's score and the share of trees "
      f"voting yes: {np.abs(ensemble - vote_share).max():.1e}")

solo = np.array([(t.predict(X_te) == y_te).mean() for t in one_forest.estimators_])
print(f"one tree alone: accuracy {solo.mean():.4f} +/- {solo.std():.4f}")
print(f"the forest:     accuracy {(one_forest.predict(X_te) == y_te).mean():.4f}")


Every tree outputs 0 or 1 and nothing in between, and the forest's score is the
share of trees voting yes, exactly, to the last bit.

Which means the number is a **measure of how much the trees agree**. That is not
the same quantity as the probability the image is an 8, and there is no reason
the two should be equal. A single tree here is 93% accurate, so on a hard 8
about a third of the trees get it wrong; the vote share reports that
disagreement, and the disagreement is a fact about the trees rather than about
the image.

Two things this explanation does *not* say, both because they were checked and
were false. It does not say the forest cannot reach 1.0: on three other problems
it does. And it does not say the forest needs more trees: at 1000 trees instead
of 100 the worst gap moved from +0.398 to +0.392.


## 2. The metric that sees it, and the metric that does not

Run two more models for comparison. Logistic regression, which optimises exactly
the quantity a probability should be. And Gaussian naive Bayes, as the extreme
case.


In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
p_logreg = out_of_fold(logreg)
reliability(y, p_logreg, "logistic regression")

p_nb = out_of_fold(GaussianNB())
reliability(y, p_nb, "Gaussian naive Bayes")


Three results worth separating.

**Logistic regression is close to honest, and its errors have no pattern.**
Its biggest gap is -0.093, which is larger than one of the calibrated forest's,
so do not read the size alone. Read the signs: `+ - + - - -`, alternating, and
the -0.093 sits on a band holding 24 images. That is sampling noise. The
forest's gaps at the top are `+0.318 +0.398 +0.204`, all one direction, on 163
images between them. That is bias, and bias is the thing you can fix.

Logistic regression is fitted by maximising the likelihood of the labels, so
producing honest probabilities is not a pleasant side effect, it is the
objective.

**Naive Bayes is not close at all.** It puts **962** of the 1797 images above
0.9, and 17.8% of them are 8s. It said 0.999 and was right about one in six.
Brier 0.4447, against the forest's 0.0247.

**And now compare the ROC-AUCs.** The forest has the **best** ROC-AUC of the
three and the second-worst calibration. Lesson 2.7 showed ROC-AUC surviving a
model that caught nothing; here it survives a model whose numbers do not mean
what they say. ROC-AUC only asks whether positives are *ranked* above negatives.
Ranking and honesty are different properties, and only one of them lets you
set a threshold from a cost.

The Brier score is the one that sees it: mean squared error between the score
and the outcome, so it charges you for being wrong about the number as well as
wrong about the order.


## 3. Fixing it: two rescalings

Both fixes learn a function from score to honest probability, on held-back data,
and leave the model alone.

**Platt scaling** (`method="sigmoid"`) fits a two-parameter logistic curve. Two
parameters, so it needs little data and cannot do anything strange.

**Isotonic regression** (`method="isotonic"`) fits any non-decreasing step
function. More flexible, so it fixes more shapes and needs more data.

`CalibratedClassifierCV` does the cross-validation internally. Note that it is
wrapped in `cross_val_predict` here as well, so the reported numbers are from an
outer fold the calibrator never saw either.


In [ ]:
platt = CalibratedClassifierCV(forest, method="sigmoid", cv=5)
iso = CalibratedClassifierCV(forest, method="isotonic", cv=5)

p_platt = out_of_fold(platt)
p_iso = out_of_fold(iso)

reliability(y, p_platt, "forest + Platt scaling")
reliability(y, p_iso, "forest + isotonic regression")


Isotonic brings every gap inside 0.05, and Brier falls from 0.0247 to 0.0151.
Platt gets Brier to the same 0.0151 but leaves a +0.175 band, because a
two-parameter sigmoid cannot bend into the shape the forest needs.

Look at the top band. The raw forest had **7** images above 0.9. Isotonic has
**119**, and all of them are 8s. Calibration did not make the model better at
recognising 8s, it gave it permission to say so.

And the ROC-AUC barely moves, 0.9919 to 0.9928. Both rescalings are monotonic by
construction, so they cannot change the ranking. That is the reassurance: you
are not trading accuracy for honesty.


In [ ]:
# Does calibration help a model that was already calibrated? Check rather than
# assume, because "calibrate everything" is the advice you will hear.
for name, est in (("logistic regression, raw", logreg),
                  ("logistic regression + Platt",
                   CalibratedClassifierCV(logreg, method="sigmoid", cv=5)),
                  ("logistic regression + isotonic",
                   CalibratedClassifierCV(logreg, method="isotonic", cv=5))):
    p = out_of_fold(est)
    print(f"{name:<32} Brier {brier_score_loss(y, p):.4f}"
          f"   ROC-AUC {roc_auc_score(y, p):.4f}")


Calibrating logistic regression makes it slightly **worse**: Brier 0.0285 to
0.0299 with Platt. The rescaling is fitted on a subset, so it adds variance and
has no bias to remove.

So the rule is not "always calibrate". It is: **measure the gap first, and
calibrate the models that have one.** Forests, boosted trees, SVMs and naive
Bayes usually do. Logistic regression usually does not.


## 4. Now put a price on each mistake

Everything so far has been about whether the number is honest. Here is why that
matters.

Suppose a missed 8 costs 20 times what a false alarm costs. Total cost is then
`false alarms + 20 x missed`, and it depends entirely on the threshold. The
textbook answer for where to put it is one line:

```
t* = C_fp / (C_fp + C_fn)
```

With C_fp = 1 and C_fn = 20 that is 1/21, about 0.048. The derivation is short:
alerting is worth it when the expected cost of alerting, `(1 - p) x C_fp`, is
below the expected cost of staying quiet, `p x C_fn`. Rearranged, that is
`p > C_fp / (C_fp + C_fn)`.

Notice which quantity the derivation uses: `p`, the probability. Not the score.
They are the same thing only for a calibrated model.


In [ ]:
C_FP, C_FN = 1.0, 20.0
t_star = C_FP / (C_FP + C_FN)
grid = np.round(np.arange(0.01, 1.00, 0.01), 2)


def total_cost(p, t):
    alert = p >= t
    fp = int((alert & (y == 0)).sum())
    fn = int((~alert & (y == 1)).sum())
    return fp * C_FP + fn * C_FN, fp, fn


def threshold_study(p, label):
    costs = [total_cost(p, t)[0] for t in grid]
    best = grid[int(np.argmin(costs))]
    c_half = total_cost(p, 0.50)
    c_star = total_cost(p, t_star)
    c_best = total_cost(p, best)
    print(f"\n{label}")
    print(f"  left at 0.50      cost {c_half[0]:7.1f}   ({c_half[1]:4d} false alarms,"
          f" {c_half[2]:3d} missed)")
    print(f"  formula t*={t_star:.3f}   cost {c_star[0]:7.1f}   ({c_star[1]:4d} false alarms,"
          f" {c_star[2]:3d} missed)")
    print(f"  cheapest at {best:.2f}    cost {c_best[0]:7.1f}   ({c_best[1]:4d} false alarms,"
          f" {c_best[2]:3d} missed)")
    print(f"  the formula costs {c_star[0] / c_best[0] - 1:+.0%} over the minimum;"
          f" 0.50 costs {c_half[0] / c_best[0]:.1f}x the minimum")


print(f"a miss costs {C_FN:.0f} false alarms, so the formula says t* = {t_star:.4f}")
threshold_study(p_forest, "random forest, raw")
threshold_study(p_platt, "random forest + Platt")
threshold_study(p_iso, "random forest + isotonic")
threshold_study(p_logreg, "logistic regression (already calibrated)")


This is the whole lesson in one output block, so read it twice.

**Leaving the threshold at 0.50 is the expensive mistake.** The raw forest costs
1140 there and 164 at its own best threshold of 0.13. Seven times the minimum,
for a default nobody chose.

**And the formula only works on an honest score.** On the raw forest, t* = 0.048
costs 482 against a minimum of 164, so trusting it costs 194% over the minimum.
The formula is not broken. It is being handed a number that does not mean what
it says: the forest's 0.048 corresponds to a real probability well above 0.048,
so the formula alerts far too eagerly. 462 false alarms.

**Calibrate first and the formula becomes usable.** On the same forest, Platt
brings the penalty from 194% to 11% and isotonic to 7%. Fix the bias and the
formula starts landing where it should.

One thing not to over-read: you cannot rank two *different* models by their
worst gap. Logistic regression's worst gap is 0.093, larger than the isotonic
forest's 0.049, and its penalty is smaller, 2% against 7%. The gap that costs
you is the systematic one near your threshold, not the largest number in the
column.


In [ ]:
# Bias or noise? Print the worst gap next to how many bands lean the same way.
# All-one-direction is bias and it is what makes the formula miss.
print(f"{'model':<26} {'worst gap':>10} {'leaning up':>11} {'penalty':>9}")
for label, p in (("forest, raw", p_forest),
                 ("forest + Platt", p_platt),
                 ("forest + isotonic", p_iso),
                 ("logistic regression", p_logreg),
                 ("naive Bayes", p_nb)):
    gaps = []
    for lo, hi in zip(EDGES[:-1], EDGES[1:]):
        m = (p >= lo) & (p < hi)
        if m.any():
            gaps.append(y[m].mean() - p[m].mean())
    worst = max(gaps, key=abs)
    up = sum(g > 0 for g in gaps)
    costs = [total_cost(p, t)[0] for t in grid]
    penalty = total_cost(p, t_star)[0] / min(costs) - 1
    print(f"{label:<26} {worst:>+10.3f} {f'{up} of {len(gaps)}':>11} {penalty:>8.0%}")


The `leaning up` column separates bias from noise. Four of the raw forest's six
bands read high, and the three biggest are consecutive and at the confident end,
which is a shape rather than a wobble. Logistic regression's two-of-six with
alternating signs is what noise looks like.

Naive Bayes is the reminder that this penalty column is not a model score. Its
6% looks respectable and it is the worst model here by a distance: it is so
uniformly wrong that every threshold costs about the same, so t* cannot miss by
much. A small penalty means "t* is near the best available threshold", not
"this model is any good". Read it beside the Brier score, never instead of it.


## 5. The other lever: weight the classes during training

Moving the threshold is a decision made after the model exists. The alternative
is telling the model about the cost while it trains, with `class_weight`.

These are not the same operation and it is worth knowing whether you need both.


In [ ]:
for cw in (None, "balanced", {0: 1, 1: 20}):
    m = RandomForestClassifier(random_state=SEED, n_jobs=1, class_weight=cw)
    p = out_of_fold(m)
    costs = [total_cost(p, t)[0] for t in grid]
    best = grid[int(np.argmin(costs))]
    print(f"class_weight={str(cw):<16} cost at 0.50 {total_cost(p, 0.50)[0]:7.1f}"
          f"   cheapest {total_cost(p, best)[0]:7.1f} at t={best:.2f}")


Weighting helps at the default threshold: 1140 down to 800 with a 20-to-1
weight. It does not get you anywhere near the 118 that the same weighted model
reaches once you also move the threshold.

So they are complementary, not alternatives. Weighting shifts what the model
learns; the threshold shifts what you do with what it learned. The cheapest
configuration measured here uses both.


## What to take from this

| Claim | What we measured |
|---|---|
| A model's 0.6 means 60% | Not for the forest. It said 0.602 and 100% of those were 8s |
| ROC-AUC tells you the scores are trustworthy | No. The forest had the best ROC-AUC and said 0.602 where the truth was 1.000 |
| Calibration costs you accuracy | No. Monotonic rescaling cannot change the ranking. AUC 0.9919 to 0.9928 |
| You should calibrate every model | No. It made logistic regression slightly worse. Measure the gap first |
| 0.5 is a reasonable default threshold | It cost 7x the minimum here |
| The cost formula t* gives you the threshold | Only on a calibrated score. On the raw forest it cost 194% over the minimum |
| class_weight replaces threshold tuning | No. Both together beat either alone |

The habit to carry: **before you choose a threshold, check that the score means
what it says.** It takes one reliability table, and lesson 2.7's arithmetic is
only as good as the number you feed it.


## Try this

1. Change `C_FN` to 5, then to 100. Watch t* move and watch the gap between
   t* and the empirical minimum stay roughly proportional to the calibration
   gap rather than to the cost ratio.
2. Run the reliability table on `p_nb` after Platt scaling. Does rescaling
   rescue a model that was ranking badly to begin with?
3. Replace the forest with `HistGradientBoostingClassifier` from lesson 2.6.
   Niculescu-Mizil and Caruana found boosted trees pushed *away* from 0 and 1,
   which is the same direction as this forest, so predict the sign of the gap
   column before you run it.
4. Run the reliability table on `load_breast_cancer`, which is 63% positive
   rather than 9.7%. The forest's worst gap there is negative, not positive.
   The lesson is that the direction is a property of the problem as much as of
   the model, which is why you measure it instead of looking it up.
